In [ ]:
# 拓展活动：寻找最优模型，并保存

In [12]:
import numpy as np  #数据库
import keras
import tensorflow as tf
#从keras库导入处理神经网络模型model所需的函数。
from keras.models import Sequential   #序贯模型
from keras.models import load_model 
#从keras库导入设计神经网络模型的各个层次layers需要用到的函数。
from keras.applications.inception_v3 import InceptionV3, preprocess_input
from keras.layers import Dense,Dropout,Activation,Flatten  #构建 keras网络结构
from keras.layers import Conv2D, MaxPooling2D
#从keras库导入神经网络模型的优化器
from keras.optimizers import SGD   #SGD随机梯度下降

%matplotlib inline
import cv2
import matplotlib.pyplot as plt    #画图

In [2]:
#下面的函数get_nb_files（）功能是：获取指定目录directory下面的所有文件的总个数
import os
import glob
def get_nb_files(directory):
  """Get number of files by searching directory recursively"""
  if not os.path.exists(directory):
    return 0
  cnt = 0
  for r, dirs, files in os.walk(directory):
    for dr in dirs:
      cnt += len(glob.glob(os.path.join(r, dr + "/*")))
  return cnt

In [6]:
train_dir = 'hw//train'
val_dir = 'hw//train'
output_model_file = 'newModel1.h5'
nb_train_samples = get_nb_files(train_dir)      # 训练样本个数
nb_val_samples = get_nb_files(val_dir)       #测试集样本个数
 
print('用于训练的样本总数nb_train_samples:',nb_train_samples,' 用于测试检验的样本总数nb_val_samples:',nb_val_samples)

#模型训练的参数,，跟输入图像的分类数目有关，跟图像的尺寸也有关。
# nb_classes= 10 #分类数，此行可以省略，因为下面一行会根据子文件夹的个数自动计算分类数。
nb_classes = len(glob.glob(train_dir + "/*"))   #我的分类总数，就是训练文件夹train和测试文件夹test下面的子文件夹的个数
print("分类数nb_classes:",nb_classes)
#IM_WIDTH, IM_HEIGHT = 299, 299  #若输入图像尺寸为28*28这种小图片，那么需要调小。
IM_WIDTH, IM_HEIGHT = 28, 28

batch_size = 512 #batch_size一般设置为总文件数的1%左右，如果太小，训练速度会很慢。
nb_epoch = 200

用于训练的样本总数nb_train_samples: 3009  用于测试检验的样本总数nb_val_samples: 3009
分类数nb_classes: 26


In [13]:
#数据准备：使用图片生成器ImageDataGenerator从原始数据生成更多数据
#　图片生成器ImageDataGenerator,通过拉伸，平移，旋转等使得数据量增大
from keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
     rescale = 1./255,#!!!此行最好不省略。下列4行的数据变换是否有，基本不会增加训练耗时
  
     rotation_range=20,  #图片随机转动的角度
     width_shift_range=0.2, #图片随机水平偏移的幅度
     height_shift_range=0.2, #图片随机竖直偏移的幅度
     fill_mode='nearest',  #当进行变换时超出边界的点将根据本参数给定的方法进行处理
     horizontal_flip=False) #适用于水平翻转不影响图片语义的时候

test_datagen = ImageDataGenerator(
     rescale = 1./255)  #!!!此行最好不省略。

# 从所给的原始的文件夹 生成训练数据train_generator与测试数据validation_generator
train_generator = train_datagen.flow_from_directory(
train_dir,  #原始的train文件夹
target_size=(IM_WIDTH, IM_HEIGHT),
batch_size=batch_size,
class_mode='categorical')
 
validation_generator = test_datagen.flow_from_directory(
val_dir,    #原始的test文件夹
target_size=(IM_WIDTH, IM_HEIGHT),
batch_size=batch_size,
class_mode='categorical')

Found 3009 images belonging to 26 classes.
Found 3009 images belonging to 26 classes.


In [5]:
#mnist的模型设计和训练
#从keras库导入设计神经网络模型的各个层次layers需要用到的函数。
from keras.layers import Dense,Dropout,Activation,Flatten  #构建 keras网络结构
from keras.layers import Conv2D, MaxPooling2D
model= Sequential() #创建一个序贯Sequential类型的模型，取名为model,接下来为model增加几层节点。

model.add(Conv2D(64,(3,3),input_shape = (28,28,3)))#卷积层
model.add(Activation('relu'))                         
model.add(MaxPooling2D(pool_size=(2,2)))#池化层
#以上3行的结构可以重复出现多次， Conv2D(参数可以改)，Activation('relu')，MaxPooling2D(参数可以改)
model.add(Conv2D(256,(3,3)))
model.add(Activation('relu'))
model.add(Flatten()) #快要结束前的一层Flatten()

model.add(Dropout(0.5))

model.add(Dense(256))  #放置在输出层之前，可以修改节点数目
model.add(Activation('relu'))

model.add(Dropout(0.5))

model.add(Dense(26)) #输出层
model.add(Activation('softmax'))

model.summary()
#编译模型方案1，优化器选rmsprop
model.compile(optimizer='rmsprop',loss='categorical_crossentropy', metrics=['accuracy'])
print("数据导入和预处理已经完成，模型设计完毕")

Instructions for updating:
Colocations handled automatically by placer.
Instructions for updating:
Please use `rate` instead of `keep_prob`. Rate should be set to `rate = 1 - keep_prob`.
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv2d_1 (Conv2D)            (None, 26, 26, 64)        1792      
_________________________________________________________________
activation_1 (Activation)    (None, 26, 26, 64)        0         
_________________________________________________________________
max_pooling2d_1 (MaxPooling2 (None, 13, 13, 64)        0         
_________________________________________________________________
conv2d_2 (Conv2D)            (None, 11, 11, 256)       147712    
_________________________________________________________________
activation_2 (Activation)    (None, 11, 11, 256)       0         
_________________________________________________________________
flatten_1 (Flatten)  

In [ ]:
'''
#MLP多层神经网络模型设计
model= Sequential() #创建一个序贯Sequential类型的模型，取名为model,接下来为model增加几层节点。
model.add(Dense(input_dim=784,units=784,activation='relu'))   #增加add输入层Dense，该层节点数目units为784
model.add(Dense(            units=50,activation='relu'))   #增加add隐藏层Dense，该层节点数目units为50
model.add(Dense(            units=10,activation='softmax'))
#model.summary()
#编译模型
sgd = SGD(lr=0.01, decay=1e-6, momentum=0.9, nesterov=True) # 优化函数，设定学习率（lr）等参数
model.compile(optimizer=sgd,loss='categorical_crossentropy', metrics=['accuracy']) 
print("数据导入和预处理已经完成，模型设计完毕")
'''

# 训练200次，rescale = 1./255,不是featurewise_center=True,和featurewise_std_normalization=True。速度快很多。训练到136次的时候，耗时差不多8小时。准确率差不多到了99%。
![image.png](attachment:image.png)

In [ ]:
#以下是模型训练的参数


import time 
t1 = time.strftime("%Y/%m/%d  %H:%M:%S")
print("模型训练开始时间starting time:",t1) ##24小时格式 
# 模型训练
history_ft = model.fit_generator(
train_generator,
steps_per_epoch=60000/batch_size,
nb_epoch=nb_epoch,
validation_data=validation_generator,
validation_steps=60000/batch_size,
class_weight='auto1',verbose=1) #verbose=2的意思：每个epoch没有时间戳？

In [ ]:
import time
t2 = time.strftime("%Y/%m/%d  %H:%M:%S")
print("模型训练开始和结束时间分别为 ：", t1, t2)
# 模型保存
# model.save("newModel1.h5",t2) 

In [ ]:
#acc
import matplotlib.pyplot as plt
plt.plot(history_ft.history['acc'])
plt.plot(history_ft.history['val_acc'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='lower right')
plt.savefig('acc.jpg')

In [ ]:
#loss
plt.plot(history_ft.history['loss'])
plt.plot(history_ft.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper right')
plt.savefig('loss.jpg')

In [7]:
from keras.preprocessing import image
from keras.models import load_model
import numpy as np
#普通用户先加载模型，
print("普通用户评估模型，测试一张图片")
model = load_model(output_model_file)
print("完成模型加载，开始使用")

普通用户评估模型，测试一张图片
Instructions for updating:
Use tf.cast instead.
完成模型加载，开始使用


In [20]:
#读入一张照片进行测试
img_path = val_dir+'/1a/a(1).jpg'
img = image.load_img(img_path, target_size=(IM_WIDTH, IM_HEIGHT))
x = image.img_to_array(img)
x = np.expand_dims(x, axis=0)
x = preprocess_input(x)

preds = model.predict(x)
print(preds)
max_seq=np.argmax(preds)  #找到每行最大的序号
print(max_seq)
print("使用模型对上述图片的预测结果为",max_seq)

[[2.3402716e-12 3.3095171e-26 8.0974726e-24 0.0000000e+00 0.0000000e+00
  0.0000000e+00 0.0000000e+00 0.0000000e+00 1.6220960e-37 0.0000000e+00
  0.0000000e+00 8.9367849e-34 1.5709660e-15 2.5485362e-17 4.6299742e-14
  5.6071270e-35 2.0115487e-32 1.8544044e-33 0.0000000e+00 3.9820135e-20
  0.0000000e+00 0.0000000e+00 0.0000000e+00 0.0000000e+00 0.0000000e+00
  1.0000000e+00]]
25
使用模型对上述图片的预测结果为 25


[[1.03241770e-13 0.00000000e+00 3.79128169e-27 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  7.65361104e-18 2.30342497e-21 2.40872242e-16 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 1.44980115e-36
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 1.00000000e+00]]
25
使用模型对上述图片的预测结果为 25


In [19]:
#读入一张照片进行测试
img_path = val_dir+'/26z/z(1).png'
img = image.load_img(img_path, target_size=(IM_WIDTH, IM_HEIGHT))
x = image.img_to_array(img)
x = np.expand_dims(x, axis=0)
x = preprocess_input(x)

preds = model.predict(x)
print(preds)
max_seq=np.argmax(preds)  #找到每行最大的序号
print(max_seq)
print("使用模型对上述图片的预测结果为",max_seq)

[[1.0794789e-03 2.3402694e-10 2.1004854e-15 7.1290117e-07 1.0001409e-08
  2.7577758e-09 9.8607645e-13 9.7167812e-09 1.5775657e-15 1.1449549e-06
  4.3041135e-05 6.6028286e-07 4.8660695e-08 6.8331043e-16 9.9966710e-05
  3.3786111e-07 2.9325859e-06 9.9752730e-01 9.3696564e-11 3.2760610e-11
  2.0504167e-06 3.5126923e-06 1.6902052e-13 6.4304970e-07 4.5684986e-14
  1.2381374e-03]]
17
使用模型对上述图片的预测结果为 17
